# Fetching and Analyzing Country Data Using a Public REST API

**Objective:** Retrieve live country data from the REST Countries API, parse the JSON response, and produce a brief summary.

**API Endpoint:** `https://restcountries.com/v3.1/all?fields=name,population,region,area`

---
## Task 1 — Fetch and Validate the Response

Send a GET request to the endpoint, print the HTTP status code, and display the total number of countries if successful.

In [1]:
import requests

# Using ?fields= to fetch only required fields (more efficient + avoids 400 on /all)
API_URL = "https://restcountries.com/v3.1/all?fields=name,population,region,area"

try:
    response = requests.get(API_URL, timeout=15)
    print(f"HTTP Status Code: {response.status_code}")

    if response.status_code == 200:
        data = response.json()
        print(f"Total number of countries retrieved: {len(data)}")
    else:
        print(f"Error: Request failed with status code {response.status_code}. "
              f"Reason: {response.reason}")

except requests.exceptions.ConnectionError:
    print("Error: Unable to connect to the API. Please check your internet connection.")
except requests.exceptions.Timeout:
    print("Error: The request timed out. The server may be unavailable.")
except requests.exceptions.RequestException as e:
    print(f"Error: An unexpected error occurred — {e}")

HTTP Status Code: 200
Total number of countries retrieved: 250


---
## Task 2 — Extract and Display Key Fields

Parse the JSON response to extract `common_name`, `population`, `region`, and `area` for each country. Store in a Pandas DataFrame and display the first 10 rows.

In [5]:
import pandas as pd

# Extract relevant fields from the JSON data
records = []
for country in data:
    records.append({
        "common_name": country.get("name", {}).get("common", None),
        "population":  country.get("population", None),
        "region":      country.get("region", None),
        "area":        country.get("area", None)
    })

# Build DataFrame
df = pd.DataFrame(records)

print(f"DataFrame shape: {df.shape[0]} rows × {df.shape[1]} columns")
print("\nFirst 10 rows:")
df.head(10)

DataFrame shape: 250 rows × 4 columns

First 10 rows:


,common_name,population,region,area
0,Cook Islands,15040,Oceania,236.0
1,Guinea,14363931,Africa,245857.0
2,Christmas Island,1692,Oceania,135.0
3,Togo,8095498,Africa,56785.0
4,Taiwan,23317031,Asia,36197.0
5,Kyrgyzstan,7281800,Asia,199951.0
6,Suriname,616500,Americas,163820.0
7,Dominican Republic,10771504,Americas,48671.0
8,Guatemala,18079810,Americas,108889.0
9,Algeria,47400000,Africa,2381741.0


---
## Task 3 — Summarize the Data

Using the DataFrame, answer:
1. Which region has the highest total population?
2. Which country has the largest area?

In [3]:
# --- Q1: Region with the highest total population ---
region_population = (
    df.groupby("region", dropna=True)["population"]
    .sum()
    .sort_values(ascending=False)
)

top_region = region_population.idxmax()
top_region_pop = region_population.max()

print("Region with the highest total population:")
print(f"  → {top_region} ({top_region_pop:,} people)")
print()

# --- Q2: Country with the largest area ---
largest_area_idx = df["area"].idxmax()
largest_country  = df.loc[largest_area_idx, "common_name"]
largest_area_val = df.loc[largest_area_idx, "area"]

print("Country with the largest area:")
print(f"  → {largest_country} ({largest_area_val:,.0f} km²)")

Region with the highest total population:
  → Asia (4,724,731,966 people)

Country with the largest area:
  → Russia (17,098,246 km²)
